# Customer analysis - 2M rows

Same idea as the 100k notebook but for the ~350mb file. `pandas.read_csv()` on that
file needs a few GB of ram because every string becomes a python object, so instead
the file is read line by line with the `csv` module and only small counters are kept
in memory. The rows themselves are thrown away immediately.

In [1]:
import csv
import sys
import time
import resource
from datetime import date
from collections import Counter

import pandas as pd
from IPython.display import display

FILE = "data/customers-2000000.csv"

# Company / City / email domain have millions of different values. keeping all
# of them in a dict is exactly the thing i'm trying to avoid, so when a counter
# gets bigger than this i drop the long tail and keep the popular ones.
# the top 10 stays correct, the rare counts don't.
MAX_KEYS = 200000

DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

pd.set_option("display.max_rows", 200)


def peak_mem_mb():
    m = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    # mac returns bytes, linux returns kilobytes
    if sys.platform == "darwin":
        return m / 1024.0 / 1024.0
    return m / 1024.0


def prune(counter):
    if len(counter) <= MAX_KEYS:
        return
    keep = dict(counter.most_common(MAX_KEYS // 2))
    counter.clear()
    counter.update(keep)


def as_table(pairs, total, name="count"):
    # (key, count) pairs -> small table with a percentage column
    out = pd.DataFrame(list(pairs), columns=["", name]).set_index("")
    out["% of rows"] = (out[name] / total * 100).round(2)
    return out

## Stream the file

This is the slow cell - it reads every row once and fills the counters.

In [2]:
start = time.time()

total = 0
bad_rows = 0
empty = Counter()
countries = Counter()
cities = Counter()
companies = Counter()
domains = Counter()
tlds = Counter()
first_names = Counter()
last_names = Counter()
months = Counter()
years = Counter()
weekdays = Counter()
with_ext = 0
with_plus = 0
min_date = None
max_date = None

seen_ids = set()
seen_emails = set()
dup_ids = 0
dup_emails = 0

f = open(FILE, newline="", encoding="utf-8")
reader = csv.reader(f)
header = next(reader)

# look up the column positions once instead of doing it for every row
col = {}
for i, name in enumerate(header):
    col[name] = i

i_country = col["Country"]
i_city = col["City"]
i_company = col["Company"]
i_email = col["Email"]
i_first = col["First Name"]
i_last = col["Last Name"]
i_date = col["Subscription Date"]
i_phone = col["Phone 1"]
i_id = col["Customer Id"]

print("streaming", FILE, "...")

for row in reader:
    total += 1

    if len(row) != len(header):
        bad_rows += 1
        continue

    # checking every column for blanks on every row is slow, so only do the
    # per column loop when there is actually a blank somewhere in the row
    if "" in row:
        for name, i in col.items():
            if row[i] == "":
                empty[name] += 1

    countries[row[i_country]] += 1
    cities[row[i_city]] += 1
    companies[row[i_company]] += 1
    first_names[row[i_first]] += 1
    last_names[row[i_last]] += 1

    email = row[i_email]
    at = email.rfind("@")
    if at != -1:
        d = email[at + 1:].lower()
        domains[d] += 1
        tlds[d.rsplit(".", 1)[-1]] += 1

    # dates are all yyyy-mm-dd so i can slice instead of parsing strings
    sub = row[i_date]
    if len(sub) == 10:
        years[sub[:4]] += 1
        months[sub[:7]] += 1
        weekdays[date(int(sub[0:4]), int(sub[5:7]), int(sub[8:10])).weekday()] += 1
        if min_date is None or sub < min_date:
            min_date = sub
        if max_date is None or sub > max_date:
            max_date = sub

    phone = row[i_phone]
    if "x" in phone:
        with_ext += 1
    if phone.startswith("+"):
        with_plus += 1

    cid = row[i_id]
    if cid in seen_ids:
        dup_ids += 1
    else:
        seen_ids.add(cid)
    if email in seen_emails:
        dup_emails += 1
    else:
        seen_emails.add(email)

    if total % 250000 == 0:
        prune(cities)
        prune(companies)
        prune(domains)
        print(f"  {total:>9,} rows   {peak_mem_mb():>5.0f} mb   {time.time() - start:>4.0f} s")

f.close()
read_time = time.time() - start

pd.Series({
    "file": FILE,
    "rows": total,
    "columns": len(header),
    "read time (s)": round(read_time, 1),
    "rows/sec": round(total / read_time),
}, name="value").to_frame()

streaming customers-2000000.csv ...
    250,000 rows     199 mb      4 s
    500,000 rows     317 mb      9 s
    750,000 rows     429 mb     13 s
  1,000,000 rows     452 mb     17 s
  1,250,000 rows     488 mb     22 s
  1,500,000 rows     646 mb     26 s
  1,750,000 rows     696 mb     31 s
  2,000,000 rows     714 mb     35 s


,value
file,customers-2000000.csv
rows,2000000
columns,12
read time (s),35.2
rows/sec,56774


## Data quality

In [3]:
if bad_rows:
    print(f"rows with the wrong number of columns: {bad_rows}")
else:
    print(f"all rows have the expected {len(header)} columns")

if empty:
    display(as_table(empty.most_common(), total, "empty fields"))
else:
    print("no empty fields anywhere")

display(pd.Series({
        "duplicated Customer Id": dup_ids,
        "duplicated Email": dup_emails,
}, name="rows").to_frame())

all rows have the expected 12 columns
no empty fields anywhere


,rows
duplicated Customer Id,0
duplicated Email,1250


## Where the customers are

In [4]:
display(as_table(countries.most_common(10), total).style.set_caption("top 10 countries"))
display(as_table(countries.most_common()[-5:], total).style.set_caption("bottom 5 countries"))

top10 = sum(c for _, c in countries.most_common(10))
pd.Series({
    "countries": len(countries),
    "top 10 share of all customers (%)": round(top10 * 100.0 / total, 1),
    "average per country": round(total / len(countries)),
}, name="value").to_frame()

,count,% of rows
,,
Korea,16240,0.810000
Congo,16208,0.810000
Jordan,8428,0.420000
Vietnam,8388,0.420000
Suriname,8375,0.420000
Timor-Leste,8372,0.420000
Zimbabwe,8360,0.420000
Slovakia (Slovak Republic),8359,0.420000
Netherlands,8354,0.420000


,count,% of rows
,,
French Guiana,7976,0.400000
Eritrea,7974,0.400000
Iceland,7965,0.400000
Slovenia,7964,0.400000
Nicaragua,7958,0.400000


,value
countries,243.0
top 10 share of all customers (%),5.0
average per country,8230.0


In [5]:
print(f"{len(cities)} cities kept in memory (the long tail was dropped)")
as_table(cities.most_common(10), total)

119928 cities kept in memory (the long tail was dropped)


,count,% of rows
,,
Bradleymouth,163,0.01
Leemouth,156,0.01
Ashleymouth,155,0.01
Kirkmouth,153,0.01
Barrymouth,151,0.01
Allenmouth,149,0.01
Ryanmouth,148,0.01
Frankmouth,147,0.01
Leonardmouth,146,0.01


## Subscriptions over time

In [6]:
display(pd.Series({
    "first signup": min_date,
    "last signup": max_date,
}, name="value").to_frame())

as_table(sorted(years.items()), total, "signups")

,value
first signup,2020-01-01
last signup,2022-05-30


,signups,% of rows
,,
2020,831045,41.55
2021,830392,41.52
2022,338563,16.93


In [7]:
best = max(months, key=months.get)
worst = min(months, key=months.get)

print(f"{len(months)} months")
display(pd.Series({
    f"best month ({best})": months[best],
    f"worst month ({worst})": months[worst],
}, name="signups").to_frame())

as_table(sorted(months.items()), total, "signups")

29 months


,signups
best month (2020-10),70780
worst month (2022-02),63671


,signups,% of rows
,,
2020-01,70264,3.51
2020-02,65831,3.29
2020-03,70477,3.52
2020-04,68109,3.41
2020-05,70545,3.53
2020-06,68144,3.41
2020-07,70365,3.52
2020-08,70111,3.51
2020-09,67861,3.39


In [8]:
as_table([(DAYS[i], weekdays[i]) for i in range(7)], total, "signups")

,signups,% of rows
,,
Monday,283817,14.19
Tuesday,284229,14.21
Wednesday,286275,14.31
Thursday,285299,14.26
Friday,286541,14.33
Saturday,287309,14.37
Sunday,286530,14.33


## Email / company

In [9]:
print(f"{len(domains)} email domains kept in memory")
display(as_table(domains.most_common(10), total).style.set_caption("top 10 email domains"))
display(as_table(tlds.most_common(8), total).style.set_caption("top email TLDs"))

178478 email domains kept in memory


,count,% of rows
,,
rojas.com,898,0.040000
benson.com,889,0.040000
huang.com,886,0.040000
valentine.com,873,0.040000
schmidt.com,870,0.040000
wolfe.com,869,0.040000
barrera.com,869,0.040000
huynh.com,867,0.040000
tapia.com,867,0.040000


,count,% of rows
,,
com,1199737,59.990000
info,200916,10.050000
org,200234,10.010000
biz,199729,9.990000
net,199384,9.970000


In [10]:
print(f"{len(companies)} companies kept in memory")
as_table(companies.most_common(10), total)

100000 companies kept in memory


,count,% of rows
,,
Bridges Group,152,0.01
Conway LLC,148,0.01
Cooke and Sons,147,0.01
Shields Ltd,146,0.01
Oneill Ltd,146,0.01
Cabrera Group,145,0.01
Clarke Ltd,144,0.01
Dunn and Sons,144,0.01
Moses LLC,143,0.01


## Names

In [11]:
print(f"{len(first_names)} distinct first names, {len(last_names)} distinct last names")
display(as_table(first_names.most_common(10), total).style.set_caption("top 10 first names"))
display(as_table(last_names.most_common(10), total).style.set_caption("top 10 last names"))

690 distinct first names, 1000 distinct last names


,count,% of rows
,,
Mariah,3082,0.150000
Samantha,3040,0.150000
Jonathon,3037,0.150000
Patricia,3031,0.150000
Kerry,3027,0.150000
Nancy,3026,0.150000
Jenny,3023,0.150000
Henry,3022,0.150000
Jackson,3022,0.150000


,count,% of rows
,,
Bird,2133,0.110000
Gill,2133,0.110000
Hubbard,2126,0.110000
Lara,2119,0.110000
Lamb,2115,0.110000
Blevins,2109,0.110000
Craig,2109,0.110000
Becker,2107,0.110000
Porter,2106,0.110000


## Phone numbers

In [12]:
phones = pd.DataFrame(
    {"rows": [with_ext, with_plus]},
    index=["Phone 1 has an extension (x)", "Phone 1 has a country code (+)"],
)
phones["% of rows"] = (phones["rows"] / total * 100).round(1)
phones

,rows,% of rows
Phone 1 has an extension (x),1199354,60.0
Phone 1 has a country code (+),320160,16.0


## Done

In [13]:
print(f"done in {time.time() - start:.1f} seconds, peak memory {peak_mem_mb():.0f} mb")

done in 35.6 seconds, peak memory 716 mb
